# House Price Prediction
**Dataset:** House Prices — Advanced Regression Techniques (Kaggle)  
**Author:** Fikri Firstly Arrasyid Hawe  
**Goal:** Predict house sale prices using feature engineering and XGBoost regression.

---
### Setup
Run `pip install kagglehub pandas scikit-learn xgboost matplotlib seaborn` before starting.

In [ ]:
import kagglehub
import os, pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
import xgboost as xgb

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

path = kagglehub.competition_download('house-prices-advanced-regression-techniques')
df_train = pd.read_csv(os.path.join(path, 'train.csv'))
print(f'Train shape: {df_train.shape}')
df_train[['SalePrice', 'GrLivArea', 'OverallQual', 'YearBuilt']].describe()

## 1. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Sale price distribution
sns.histplot(df_train['SalePrice'], kde=True, color='#c8a87a', ax=axes[0,0])
axes[0,0].set_title('Sale Price Distribution')

# Log-transformed
sns.histplot(np.log1p(df_train['SalePrice']), kde=True, color='#1a1a1a', ax=axes[0,1])
axes[0,1].set_title('Log(Sale Price) Distribution')

# Overall quality vs price
sns.boxplot(data=df_train, x='OverallQual', y='SalePrice', palette='YlOrBr', ax=axes[1,0])
axes[1,0].set_title('Sale Price by Overall Quality')

# Living area vs price
axes[1,1].scatter(df_train['GrLivArea'], df_train['SalePrice'], alpha=0.4, color='#c8a87a', s=10)
axes[1,1].set_title('Living Area vs Sale Price')
axes[1,1].set_xlabel('Above Ground Living Area (sqft)')

plt.tight_layout()
plt.show()

## 2. Preprocessing & Feature Engineering

In [ ]:
df = df_train.copy()

# Target: log-transform
df['SalePrice'] = np.log1p(df['SalePrice'])

# Fill missing values
for col in df.select_dtypes(include='object').columns:
    df[col].fillna('None', inplace=True)
for col in df.select_dtypes(include='number').columns:
    df[col].fillna(df[col].median(), inplace=True)

# Feature engineering
df['HouseAge'] = df['YrSold'] - df['YearBuilt']
df['RemodelAge'] = df['YrSold'] - df['YearRemodAdd']
df['TotalBath'] = df['FullBath'] + 0.5 * df['HalfBath'] + df['BsmtFullBath'] + 0.5 * df['BsmtHalfBath']
df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']

# Encode categoricals
le = LabelEncoder()
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])

X = df.drop(columns=['SalePrice', 'Id'])
y = df['SalePrice']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Features: {X.shape[1]}, Train: {len(X_train)}, Val: {len(X_val)}')

## 3. XGBoost Model

In [ ]:
model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    early_stopping_rounds=30,
    eval_metric='rmse',
    verbosity=0
)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

y_pred = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print(f'Validation RMSE (log scale): {rmse:.4f}')
print(f'Approximate price RMSE: ${np.expm1(rmse):,.0f}')

In [ ]:
# Feature importance
importance = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=True).tail(15)
importance.plot(kind='barh', color='#c8a87a', figsize=(10, 6))
plt.title('Top 15 Feature Importances (XGBoost)')
plt.tight_layout()
plt.show()

## 4. Conclusions

- **XGBoost with feature engineering** achieves strong prediction performance on house prices
- **OverallQual, TotalSF, and GrLivArea** are the most predictive features
- Log-transforming the target variable significantly improves model accuracy
- **Engineered features** (HouseAge, TotalSF, TotalBath) improve predictions meaningfully over raw features